# 2-Model Ensemble: TF-IDF + Flan-T5-Large
## Structured Data Extraction from Rectal Cancer MRI Reports

**Pipeline:** TF-IDF baseline → Flan-T5-Large fine-tuning → Per-field smart ensemble → Rule-based constraint layer

This notebook is **fully self-contained** — no reference to prior notebooks needed.  
Upload `train.csv`, `valid.csv`, and `test.csv` to `/content/` before running.

In [ ]:
# Install required packages (Colab-compatible)
!pip install -q numpy==2.0.2 pandas==2.2.2 scikit-learn==1.5.2 \
    transformers==4.41.2 datasets==2.19.1 accelerate==0.30.1 sentencepiece

## Step 0: Import Libraries & Set Random Seeds

In [ ]:
import ast, json, math, random, re, os, copy, platform
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Step 1: Load Data & Define the 54-Field Extraction Schema

Each CSV has columns `id`, `text` (free-text MRI report), and `output` (nested dict as string).  
We flatten the nested structure into 54 leaf fields: 41 categorical + 13 numeric.

In [ ]:
# ------------------------------------------------------------------
# Load CSVs (adjust paths if not on Colab)
# ------------------------------------------------------------------
train_df = pd.read_csv('/content/train.csv')
valid_df = pd.read_csv('/content/valid.csv')
test_df  = pd.read_csv('/content/test.csv')
print('train:', train_df.shape, 'valid:', valid_df.shape, 'test:', test_df.shape)

# ------------------------------------------------------------------
# Schema constants — define the canonical field ordering
# ------------------------------------------------------------------
MISSING_MARKERS = {'Not Mentioned', 'Not Applicable', 'Not Relevant', 'Nil Significant'}

TOP_ORDER = [
    'report_type','bio','mri_numeric','mri_stage','t2_signal_intensity','dwi','morphology',
    'post_treatment_change','rectal_perforation','radial_extent','crm_or_mrf_involvement',
    'emvi','tumour_deposit','is_t4a','adjacent_structures_t4b','anal_sphincter_complex',
    't_stage','n_stage','m_stage','mr_trg'
]
BIO_ORDER = ['age','gender']
MRI_NUMERIC_ORDER = [
    'current_length_of_tumour','location_of_tumour','distance_from_anal_verge',
    'distance_from_anorectal_junction','extramural_spread_size','mr_crm_distance',
    'number_of_mesorectal_nodes','internal_iliac_nodes','size_of_obturator_nodes',
    'size_of_inguinal','size_of_external_iliac','size_of_common_iliac','size_of_para_aortic'
]
MRI_STAGE_ORDER = [
    'is_first_mri_report','is_restaging_mri_report','is_mri_report_after_neoadjuvant',
    'is_post_treatment_mri_report','is_post_operative_mri'
]
POST_TREATMENT_ORDER = [
    'is_thick_t2_hypointense_band','is_thin_t2_hypointense_band',
    'is_residual_tumor_as_first_mri','is_mucin_reaction_t2_hyper_hypo_post_treatment'
]
RECTAL_PERF_ORDER = ['obstruction','perforation']
ADJ_T4B_ORDER = {
    'male': ['prostate','seminal_vesicles'],
    'female': ['ovaries','uterus','vagina'],
    'male_and_female': ['puborectalis','levator_ani','obturator_internus','obturator_externus','piriformis'],
}
ANAL_T4B_ORDER = ['external_sphincter','inter_sphincteric_plane','ischiorectal_foss','fistula_in_ano']

# The 13 numeric fields (everything else is categorical)
NUMERIC_FIELDS = {
    'bio.age','mri_numeric.current_length_of_tumour','mri_numeric.distance_from_anal_verge',
    'mri_numeric.distance_from_anorectal_junction','mri_numeric.extramural_spread_size',
    'mri_numeric.mr_crm_distance','mri_numeric.number_of_mesorectal_nodes',
    'mri_numeric.internal_iliac_nodes','mri_numeric.size_of_obturator_nodes',
    'mri_numeric.size_of_inguinal','mri_numeric.size_of_external_iliac',
    'mri_numeric.size_of_common_iliac','mri_numeric.size_of_para_aortic',
}

# ------------------------------------------------------------------
# Helper functions for label normalisation and schema parsing
# ------------------------------------------------------------------

def normalize_common_label(s):
    # Standardise common missing-value labels and whitespace
    s = re.sub(r'\s+', ' ', str(s).strip())
    low = s.lower()
    if low == 'not relevant':   return 'Not Relevant'
    if low == 'not mentioned':  return 'Not Mentioned'
    if low == 'not applicable': return 'Not Applicable'
    if low == 'nil significant': return 'Nil Significant'
    s = re.sub(r'\bMR\s*TRG\b', 'MR TRG', s, flags=re.IGNORECASE)
    return s

def canonicalize_value(v):
    # Convert any raw value to its canonical string form
    if v is None: return 'Not Mentioned'
    if isinstance(v, float) and str(v) == 'nan': return 'Not Mentioned'
    if isinstance(v, (int, float)): return v
    s = normalize_common_label(v)
    s = re.sub(r'(\d)(mm|cm)\b', r'\1 \2', s, flags=re.IGNORECASE)
    s = re.sub(r'\bMM\b', 'mm', s)
    s = re.sub(r'\bCM\b', 'cm', s)
    return s

def parse_output(output_str):
    # Parse the nested dict string from the CSV
    return ast.literal_eval(output_str)

def ensure_schema(d):
    # Guarantee all expected sub-dicts exist
    d = dict(d)
    if 'mri_numeric' in d:
        mn = dict(d['mri_numeric'])
        if 'location_of_tumour' not in mn:
            mn['location_of_tumour'] = 'Not Mentioned'
        d['mri_numeric'] = mn
    return d

def ordered_schema_dict(d):
    # Build a fully ordered nested dict following the canonical schema
    d = ensure_schema(d)
    out = OrderedDict()
    for k in TOP_ORDER:
        v = d.get(k, None)
        if k == 'bio':
            vv = v if isinstance(v, dict) else {}
            out[k] = OrderedDict([(bk, canonicalize_value(vv.get(bk))) for bk in BIO_ORDER])
        elif k == 'mri_numeric':
            vv = v if isinstance(v, dict) else {}
            out[k] = OrderedDict([(mk, canonicalize_value(vv.get(mk))) for mk in MRI_NUMERIC_ORDER])
        elif k == 'mri_stage':
            vv = v if isinstance(v, dict) else {}
            out[k] = OrderedDict([(mk, canonicalize_value(vv.get(mk))) for mk in MRI_STAGE_ORDER])
        elif k == 'post_treatment_change':
            vv = v if isinstance(v, dict) else {}
            out[k] = OrderedDict([(mk, canonicalize_value(vv.get(mk))) for mk in POST_TREATMENT_ORDER])
        elif k == 'rectal_perforation':
            vv = v if isinstance(v, dict) else {}
            out[k] = OrderedDict([(mk, canonicalize_value(vv.get(mk))) for mk in RECTAL_PERF_ORDER])
        elif k == 'adjacent_structures_t4b':
            vv = v if isinstance(v, dict) else {}
            adj = OrderedDict()
            for group, keys in ADJ_T4B_ORDER.items():
                sub = vv.get(group, {}) if isinstance(vv.get(group), dict) else {}
                adj[group] = OrderedDict([(sk, canonicalize_value(sub.get(sk))) for sk in keys])
            out[k] = adj
        elif k == 'anal_sphincter_complex':
            vv = v if isinstance(v, dict) else {}
            t4b = vv.get('t4b', {}) if isinstance(vv.get('t4b'), dict) else {}
            asc = OrderedDict()
            asc['internal_sphincter'] = canonicalize_value(vv.get('internal_sphincter'))
            asc['t4b'] = OrderedDict([(sk, canonicalize_value(t4b.get(sk))) for sk in ANAL_T4B_ORDER])
            out[k] = asc
        else:
            out[k] = canonicalize_value(v)
    return out

def flatten_dict(d, prefix=''):
    # Recursively flatten nested dict to dot-separated keys
    items = {}
    for k, v in d.items():
        key = f'{prefix}.{k}' if prefix else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, key))
        else:
            items[key] = v
    return items

def parse_number_mm(value):
    # Extract a numeric value (in mm) from a string, handling 'cm' conversion
    if value is None: return None
    if isinstance(value, (int, float)) and not pd.isna(value): return float(value)
    s = normalize_common_label(value)
    if s in MISSING_MARKERS: return None
    low = s.lower()
    if low in {'not relevant', 'not mentioned', 'not applicable', 'nil significant'}: return None
    nums = re.findall(r'\d+(?:\.\d+)?', s)
    if not nums: return None
    x = max(float(n) for n in nums)
    if 'cm' in s.lower(): x = x * 10.0
    return x

def prepare_df(df):
    # Parse each row's nested output into a flat DataFrame
    rows = []
    for _, row in df.iterrows():
        y = ordered_schema_dict(parse_output(row['output']))
        flat = flatten_dict(y)
        rows.append({'id': row['id'], 'text': row['text'],
                     'target_json': json.dumps(y, ensure_ascii=False), **flat})
    return pd.DataFrame(rows)

# ------------------------------------------------------------------
# Build flat DataFrames and derive field lists
# ------------------------------------------------------------------
train_flat = prepare_df(train_df)
valid_flat = prepare_df(valid_df)
test_flat  = prepare_df(test_df)

ALL_FIELDS = [c for c in train_flat.columns if c not in ['id', 'text', 'target_json']]
CAT_FIELDS = [c for c in ALL_FIELDS if c not in NUMERIC_FIELDS]

print(f'Total fields: {len(ALL_FIELDS)}, Categorical: {len(CAT_FIELDS)}, Numeric: {len(NUMERIC_FIELDS)}')
print(f'train_flat: {train_flat.shape}, valid_flat: {valid_flat.shape}, test_flat: {test_flat.shape}')

## Step 2: Evaluation Metrics

Leaf-average categorical accuracy (each field weighted equally), macro F1, micro accuracy, record-level exact match, and numeric MAE/RMSE.

In [ ]:
def canon_text_label(x):
    # Lowercase + normalise for comparison
    return str(normalize_common_label(x)).strip().lower()

def evaluate_predictions(y_true_df, y_pred_df, cat_fields, numeric_fields):
    # Compute all evaluation metrics
    cat_accs, cat_f1s = [], []
    total_correct, total_count = 0, 0

    for col in cat_fields:
        yt = y_true_df[col].astype(str).map(canon_text_label).tolist()
        yp = y_pred_df[col].astype(str).map(canon_text_label).tolist()
        acc = accuracy_score(yt, yp)
        cat_accs.append(acc)
        cat_f1s.append(f1_score(yt, yp, average='macro', zero_division=0))
        total_correct += sum(a == b for a, b in zip(yt, yp))
        total_count += len(yt)

    # Numeric error (only where both true and pred are non-missing)
    abs_err, sq_err = [], []
    for col in numeric_fields:
        yt = y_true_df[col].map(parse_number_mm).tolist()
        yp = y_pred_df[col].map(parse_number_mm).tolist()
        for a, b in zip(yt, yp):
            if a is not None and b is not None:
                abs_err.append(abs(a - b))
                sq_err.append((a - b) ** 2)

    # Record-level exact match
    all_cols = cat_fields + list(numeric_fields)
    true_c, pred_c = y_true_df.copy(), y_pred_df.copy()
    for col in cat_fields:
        true_c[col] = true_c[col].astype(str).map(canon_text_label)
        pred_c[col] = pred_c[col].astype(str).map(canon_text_label)
    for col in numeric_fields:
        true_c[col] = true_c[col].astype(str)
        pred_c[col] = pred_c[col].astype(str)
    em = (true_c[all_cols].values == pred_c[all_cols].values).all(axis=1).mean()

    return {
        'leaf_avg_cat_accuracy': float(np.mean(cat_accs)),
        'leaf_avg_macro_f1':     float(np.mean(cat_f1s)),
        'micro_cat_accuracy':    float(total_correct / max(1, total_count)),
        'record_exact_match':    float(em),
        'numeric_mae_mm':        float(np.mean(abs_err)) if abs_err else None,
        'numeric_rmse_mm':       float(np.sqrt(np.mean(sq_err))) if sq_err else None,
    }

def print_metrics(name, m):
    # Pretty-print a metrics dict
    print(f'\n{name}')
    for k, v in m.items():
        print(f'  {k}: {v}')

def per_field_accuracy(yt_df, yp_df, fields):
    # Per-field accuracy breakdown, sorted worst-first
    rows = []
    for col in fields:
        yt = yt_df[col].astype(str).str.strip().str.lower()
        yp = yp_df[col].astype(str).str.strip().str.lower()
        acc = (yt == yp).mean()
        rows.append({'field': col, 'accuracy': round(acc, 3), 'n_errors': int((yt != yp).sum())})
    return pd.DataFrame(rows).sort_values('accuracy')

def fmt(name, split, m):
    # Format a metrics dict into a summary row for comparison tables
    return {'Model': name, 'Split': split,
            'Cat Acc': round(m.get('leaf_avg_cat_accuracy', 0) or 0, 4),
            'Macro F1': round(m.get('leaf_avg_macro_f1', 0) or 0, 4),
            'Micro Acc': round(m.get('micro_cat_accuracy', 0) or 0, 4),
            'Exact Match': round(m.get('record_exact_match', 0) or 0, 4)}

print('Evaluation helpers ready.')

## Step 3: TF-IDF + Logistic Regression Baseline

Per-field classifiers on TF-IDF features (unigram + bigram, 40K features, balanced class weights).  
Numeric fields use a two-stage approach: presence classifier → Ridge regression.

In [ ]:
# ------------------------------------------------------------------
# Vectorise all report texts
# ------------------------------------------------------------------
tfidf_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=40000, min_df=1, sublinear_tf=True)
X_tr = tfidf_vec.fit_transform(train_flat['text'].astype(str))
X_va = tfidf_vec.transform(valid_flat['text'].astype(str))
X_te = tfidf_vec.transform(test_flat['text'].astype(str))

# ------------------------------------------------------------------
# Train per-field classifiers (categorical)
# ------------------------------------------------------------------
tfidf_vp = pd.DataFrame(index=valid_flat.index)
tfidf_tp = pd.DataFrame(index=test_flat.index)

for col in tqdm(CAT_FIELDS, desc='TF-IDF cat'):
    ytr = train_flat[col].astype(str)
    if ytr.nunique() < 2:
        # Only one class in training — predict that class everywhere
        tfidf_vp[col] = [ytr.iloc[0]] * X_va.shape[0]
        tfidf_tp[col] = [ytr.iloc[0]] * X_te.shape[0]
        continue
    clf = LogisticRegression(max_iter=3000, class_weight='balanced')
    clf.fit(X_tr, ytr)
    tfidf_vp[col] = clf.predict(X_va)
    tfidf_tp[col] = clf.predict(X_te)

# ------------------------------------------------------------------
# Train per-field regressors (numeric: presence flag + Ridge)
# ------------------------------------------------------------------
for col in tqdm(sorted(NUMERIC_FIELDS), desc='TF-IDF num'):
    y_num  = train_flat[col].map(parse_number_mm)
    y_pres = (~y_num.isna()).astype(int)

    # Stage 1: is the value present at all?
    pc = LogisticRegression(max_iter=3000, class_weight='balanced')
    pc.fit(X_tr, y_pres)
    vp_flag = pc.predict(X_va)
    tp_flag = pc.predict(X_te)

    # Stage 2: if present, what's the value?
    reg  = Ridge(alpha=1.0)
    mask = (~y_num.isna()).values
    if mask.sum() > 0:
        reg.fit(X_tr[mask], y_num[mask].values)
        vv = reg.predict(X_va)
        tv = reg.predict(X_te)
    else:
        vv = np.zeros(X_va.shape[0])
        tv = np.zeros(X_te.shape[0])

    tfidf_vp[col] = [f'{max(0, float(v)):.1f} mm' if p == 1 else 'Not Mentioned'
                     for p, v in zip(vp_flag, vv)]
    tfidf_tp[col] = [f'{max(0, float(v)):.1f} mm' if p == 1 else 'Not Mentioned'
                     for p, v in zip(tp_flag, tv)]

# ------------------------------------------------------------------
# Evaluate TF-IDF
# ------------------------------------------------------------------
tfidf_vm = evaluate_predictions(valid_flat, tfidf_vp, CAT_FIELDS, NUMERIC_FIELDS)
tfidf_tm = evaluate_predictions(test_flat, tfidf_tp, CAT_FIELDS, NUMERIC_FIELDS)
print_metrics('TF-IDF — VALID', tfidf_vm)
print_metrics('TF-IDF — TEST', tfidf_tm)

## Step 4: Prompt Engineering & Dataset for Flan-T5-Large

Each of the 54 fields gets its own natural-language prompt, yielding 3,510 training examples from 65 reports.  
This field-by-field approach lets the model focus on one extraction at a time.

In [ ]:
# ------------------------------------------------------------------
# Field descriptions for prompt construction
# ------------------------------------------------------------------
FIELD_DESC = {
    'report_type': 'type of imaging report',
    'bio.age': 'patient age in years',
    'bio.gender': 'patient gender (Male or Female)',
    'mri_numeric.current_length_of_tumour': 'current tumour length measurement',
    'mri_numeric.location_of_tumour': 'tumour location',
    'mri_numeric.distance_from_anal_verge': 'distance from anal verge in mm',
    'mri_numeric.distance_from_anorectal_junction': 'distance from anorectal junction in mm',
    'mri_numeric.extramural_spread_size': 'extramural spread size in mm',
    'mri_numeric.mr_crm_distance': 'circumferential resection margin distance in mm',
    'mri_numeric.number_of_mesorectal_nodes': 'number or size of mesorectal nodes',
    'mri_numeric.internal_iliac_nodes': 'internal iliac lymph node size',
    'mri_numeric.size_of_obturator_nodes': 'obturator lymph node size',
    'mri_numeric.size_of_inguinal': 'inguinal lymph node size',
    'mri_numeric.size_of_external_iliac': 'external iliac lymph node size',
    'mri_numeric.size_of_common_iliac': 'common iliac lymph node size',
    'mri_numeric.size_of_para_aortic': 'para-aortic lymph node size',
    'mri_stage.is_first_mri_report': 'whether this is the first MRI report (Yes/No)',
    'mri_stage.is_restaging_mri_report': 'whether this is a restaging MRI (Yes/No)',
    'mri_stage.is_mri_report_after_neoadjuvant': 'whether MRI is after neoadjuvant therapy (Yes/No)',
    'mri_stage.is_post_treatment_mri_report': 'whether this is a post-treatment MRI (Yes/No)',
    'mri_stage.is_post_operative_mri': 'whether this is a post-operative MRI (Yes/No)',
    't2_signal_intensity': 'T2 signal intensity finding',
    'dwi': 'diffusion-weighted imaging finding',
    'morphology': 'tumour morphology',
    'post_treatment_change.is_thick_t2_hypointense_band': 'thick T2 hypointense band (Yes/No)',
    'post_treatment_change.is_thin_t2_hypointense_band': 'thin T2 hypointense band (Yes/No)',
    'post_treatment_change.is_residual_tumor_as_first_mri': 'residual tumour as in first MRI (Yes/No)',
    'post_treatment_change.is_mucin_reaction_t2_hyper_hypo_post_treatment': 'mucin reaction T2 signal post treatment',
    'rectal_perforation.obstruction': 'rectal obstruction (Yes/No)',
    'rectal_perforation.perforation': 'rectal perforation (Yes/No)',
    'radial_extent': 'radial extent of tumour',
    'crm_or_mrf_involvement': 'CRM or mesorectal fascia involvement (Yes/No)',
    'emvi': 'extramural vascular invasion (Yes/No)',
    'tumour_deposit': 'tumour deposit (Yes/No)',
    'is_t4a': 'T4a peritoneal involvement (Yes/No)',
    'adjacent_structures_t4b.male.prostate': 'prostate involvement (Yes/No/Not Applicable)',
    'adjacent_structures_t4b.male.seminal_vesicles': 'seminal vesicles involvement (Yes/No/Not Applicable)',
    'adjacent_structures_t4b.female.ovaries': 'ovaries involvement (Yes/No/Not Applicable)',
    'adjacent_structures_t4b.female.uterus': 'uterus involvement (Yes/No/Not Applicable)',
    'adjacent_structures_t4b.female.vagina': 'vagina involvement (Yes/No/Not Applicable)',
    'adjacent_structures_t4b.male_and_female.puborectalis': 'puborectalis involvement (Yes/No)',
    'adjacent_structures_t4b.male_and_female.levator_ani': 'levator ani involvement (Yes/No)',
    'adjacent_structures_t4b.male_and_female.obturator_internus': 'obturator internus involvement (Yes/No)',
    'adjacent_structures_t4b.male_and_female.obturator_externus': 'obturator externus involvement (Yes/No)',
    'adjacent_structures_t4b.male_and_female.piriformis': 'piriformis involvement (Yes/No)',
    'anal_sphincter_complex.internal_sphincter': 'internal sphincter involvement (Yes/No)',
    'anal_sphincter_complex.t4b.external_sphincter': 'external sphincter involvement (Yes/No)',
    'anal_sphincter_complex.t4b.inter_sphincteric_plane': 'inter-sphincteric plane involvement (Yes/No)',
    'anal_sphincter_complex.t4b.ischiorectal_foss': 'ischiorectal fossa involvement (Yes/No)',
    'anal_sphincter_complex.t4b.fistula_in_ano': 'fistula in ano (Yes/No)',
    't_stage': 'tumour T stage (T0, T1, T2, T3a, T3b, T3c, T3d, T4a, T4b)',
    'n_stage': 'lymph node N stage (N0, N1, N2)',
    'm_stage': 'metastasis M stage (M0, M1)',
    'mr_trg': 'MR tumour regression grade (MR TRG 1 through MR TRG 5)',
}

MAX_TARGET_LEN = 48  # max tokens for the target answer

def make_field_prompt(report_text, field_name):
    # Build extraction prompt for a single field
    desc = FIELD_DESC.get(field_name, field_name.replace('.', ' ').replace('_', ' '))
    return (
        f'Extract the {desc} from this rectal MRI report. '
        f'If not mentioned, answer "Not Mentioned". '
        f'If not applicable, answer "Not Applicable". '
        f'If nil significant, answer "Nil Significant".\n\n'
        f'Field: {field_name}\n\n'
        f'Report: {report_text}\n\n'
        f'Answer:'
    )


class FieldByFieldDataset(Dataset):
    # PyTorch dataset: one sample per (report, field) pair

    def __init__(self, df_flat, fields, tokenizer, max_input_len=768, max_target_len=48):
        self.items = []
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len
        for _, row in df_flat.iterrows():
            text = str(row['text'])
            for field in fields:
                prompt = make_field_prompt(text, field)
                target = str(row[field])
                self.items.append((prompt, target))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        prompt, target = self.items[idx]
        enc = self.tokenizer(prompt, max_length=self.max_input_len, truncation=True,
                             padding='max_length', return_tensors='pt')
        tgt = self.tokenizer(text_target=target, max_length=self.max_target_len,
                             truncation=True, padding='max_length', return_tensors='pt')
        labels = tgt['input_ids'].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100  # ignore padding in loss
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         labels,
        }

print('Prompt builder and Dataset class ready.')
print(f'Sample prompt (first 200 chars):\n{make_field_prompt("..sample report..", "t_stage")[:200]}')

## Step 5: Fine-Tune Flan-T5-Large (780M Parameters)

Training config: batch 4 × gradient accumulation 4 = effective batch 16, LR 2e-5, linear warmup, 15 epochs with early stopping (patience 4). Requires A100 GPU.

In [ ]:
# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
MODEL_NAME  = 'google/flan-t5-large'
BATCH_SIZE  = 4          # per-device batch (reduce to 2 if OOM)
MAX_INPUT   = 768        # longer context than base to avoid truncation
EPOCHS      = 15
LR          = 2e-5
ACCUM_STEPS = 4          # effective batch = BATCH_SIZE * ACCUM_STEPS = 16

# ------------------------------------------------------------------
# Load model and tokenizer
# ------------------------------------------------------------------
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# ------------------------------------------------------------------
# Build datasets and data loaders
# ------------------------------------------------------------------
train_ds = FieldByFieldDataset(train_flat, ALL_FIELDS, tokenizer, MAX_INPUT, MAX_TARGET_LEN)
valid_ds = FieldByFieldDataset(valid_flat, ALL_FIELDS, tokenizer, MAX_INPUT, MAX_TARGET_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# ------------------------------------------------------------------
# Optimiser and scheduler
# ------------------------------------------------------------------
optimizer   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(optimizer,
                                              num_warmup_steps=int(0.06 * total_steps),
                                              num_training_steps=total_steps)

# ------------------------------------------------------------------
# Training loop with early stopping (patience=4)
# ------------------------------------------------------------------
best_val_loss = float('inf')
best_state    = None
patience      = 0
history       = []

print(f'\nTraining: {total_steps} steps, {EPOCHS} epochs\n')

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')

    for step, batch in enumerate(pbar):
        batch = {k: v.to(device) for k, v in batch.items()}
        out   = model(**batch)
        loss  = out.loss / ACCUM_STEPS
        loss.backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += out.loss.item()
        pbar.set_postfix(loss=f'{out.loss.item():.4f}')

    avg_train = total_loss / len(train_loader)

    # --- Validate ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            val_loss += model(**batch).loss.item()
    avg_val = val_loss / len(valid_loader)

    history.append({'epoch': epoch, 'train_loss': avg_train, 'val_loss': avg_val})
    print(f'Epoch {epoch} | Train: {avg_train:.4f} | Val: {avg_val:.4f}')

    # --- Checkpoint ---
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_state = copy.deepcopy(model.state_dict())
        patience = 0
        torch.save(best_state, '/content/best_flan_t5_large.pt')
        print('  -> Saved best')
    else:
        patience += 1
        print(f'  -> No improvement ({patience}/4)')
        if patience >= 4:
            print('Early stopping.')
            break

# Restore best checkpoint
model.load_state_dict(best_state)
print(f'\nBest model loaded. Val loss: {best_val_loss:.4f}')
pd.DataFrame(history)

## Step 6: Run Inference with Flan-T5-Large

Generate predictions for all 54 fields on both validation and test sets using beam search (k=3).

In [ ]:
def predict_all_fields(mdl, tok, texts, fields, batch_size=16, max_input=768, max_gen=48):
    # Run field-by-field inference for every report using beam search (k=3)
    mdl.eval()
    all_results = []
    for text in tqdm(texts, desc='Predicting'):
        row = {}
        prompts = [make_field_prompt(text, f) for f in fields]
        # Process fields in batches for efficiency
        for i in range(0, len(prompts), batch_size):
            bp = prompts[i:i + batch_size]
            bf = fields[i:i + batch_size]
            enc = tok(bp, max_length=max_input, truncation=True,
                      padding=True, return_tensors='pt').to(mdl.device)
            with torch.no_grad():
                gen = mdl.generate(**enc, max_new_tokens=max_gen,
                                   num_beams=3, early_stopping=True)
            decoded = tok.batch_decode(gen, skip_special_tokens=True)
            for field, pred in zip(bf, decoded):
                row[field] = pred.strip()
        all_results.append(row)
    return pd.DataFrame(all_results)

print('Predicting validation set...')
flan_vp_raw = predict_all_fields(model, tokenizer,
                                  valid_flat['text'].astype(str).tolist(),
                                  ALL_FIELDS, max_input=MAX_INPUT)

print('Predicting test set...')
flan_tp_raw = predict_all_fields(model, tokenizer,
                                  test_flat['text'].astype(str).tolist(),
                                  ALL_FIELDS, max_input=MAX_INPUT)

print('Done. Shapes:', flan_vp_raw.shape, flan_tp_raw.shape)

## Step 7: Post-Processing & Normalisation (v2)

Improved post-processor with field-type-aware handling: special age logic (no "mm" suffix), regex-based staging extraction, typo correction, and fuzzy matching against the training vocabulary.

In [ ]:
def normalize_prediction_v2(pred_str, field, tf):
    # Field-aware post-processing of a raw T5 prediction
    pred = str(pred_str).strip()
    pl   = pred.lower().strip()

    # Fix common typos the model generates
    typo_map = {
        'not mentiond': 'not mentioned', 'not mentioneds': 'not mentioned',
        'not mention': 'not mentioned',  'notmentioned': 'not mentioned',
        'not aplicable': 'not applicable', 'nil signficant': 'nil significant',
        'nil significan': 'nil significant', 'nill significant': 'nil significant',
    }
    if pl in typo_map:
        pl = typo_map[pl]

    # Standard missing-value markers
    mm = {
        'not mentioned': 'Not Mentioned', 'not applicable': 'Not Applicable',
        'not relevant': 'Not Relevant',   'nil significant': 'Nil Significant',
        'nil': 'Nil Significant', 'none': 'Not Mentioned', 'n/a': 'Not Applicable',
        'na': 'Not Applicable', 'unknown': 'Not Mentioned',
        'no': 'No', 'yes': 'Yes', 'male': 'Male', 'female': 'Female',
    }
    if pl in mm:
        return mm[pl]

    # --- SPECIAL: bio.age — return integer, never "X mm" ---
    if field == 'bio.age':
        nums = re.findall(r'\d+', pred)
        if nums:
            age = int(nums[0])
            if 1 <= age <= 120:
                return str(age)
        return pred

    # --- SPECIAL: staging fields with known label sets ---
    if field == 't_stage':
        t_stages = ['T0','T1','T2','T3a','T3b','T3c','T3d','T4a','T4b']
        for ts in t_stages:
            if ts.lower() in pl or pl == ts.lower():
                return ts
        t_match = re.search(r't\s*(\d[a-d]?)', pl, re.IGNORECASE)
        if t_match:
            return 'T' + t_match.group(1)

    if field == 'n_stage':
        n_stages = ['N0','N1','N1a','N1b','N1c','N2','N2a','N2b']
        for ns in n_stages:
            if ns.lower() in pl:
                return ns
        n_match = re.search(r'n\s*(\d[a-c]?)', pl, re.IGNORECASE)
        if n_match:
            return 'N' + n_match.group(1)

    if field == 'm_stage':
        if 'm1' in pl: return 'M1'
        if 'm0' in pl: return 'M0'

    if field == 'mr_trg':
        trg_match = re.search(r'(?:mr\s*)?trg\s*(\d)', pl, re.IGNORECASE)
        if trg_match:
            return f'MR TRG {trg_match.group(1)}'

    # Exact case-insensitive match against training vocabulary
    known = {v.strip().lower(): v for v in tf[field].astype(str).unique()}
    if pl in known:
        return known[pl]

    # Substring match (longer keys first to prefer specific matches)
    for kl, kv in sorted(known.items(), key=lambda x: -len(x[0])):
        if len(kl) > 3 and (kl in pl or pl in kl):
            return kv

    # Numeric fields (except age): extract number + "mm"
    if field in NUMERIC_FIELDS and field != 'bio.age':
        nums = re.findall(r'\d+(?:\.\d+)?', pred)
        if nums:
            val = float(nums[0])  # take first number (most likely the measurement)
            if 'cm' in pl:
                val *= 10
            return f'{int(val)} mm' if val == int(val) else f'{val} mm'

    # Token-overlap fallback
    best, best_s = pred, 0
    pt = set(pl.split())
    for kl, kv in known.items():
        ov = len(pt & set(kl.split()))
        if ov > best_s:
            best_s = ov
            best = kv
    return best if best_s > 0 else pred


def postprocess_v2(pred_df, tf, fields):
    # Apply improved normalisation to every field
    r = pred_df.copy()
    for f in fields:
        r[f] = r[f].apply(lambda x: normalize_prediction_v2(x, f, tf))
    return r

# ------------------------------------------------------------------
# Apply post-processing to raw Flan-T5-Large predictions
# ------------------------------------------------------------------
flan_vp = postprocess_v2(flan_vp_raw, train_flat, ALL_FIELDS)
flan_tp = postprocess_v2(flan_tp_raw, train_flat, ALL_FIELDS)

# ------------------------------------------------------------------
# Evaluate Flan-T5-Large alone
# ------------------------------------------------------------------
flan_vm = evaluate_predictions(valid_flat, flan_vp, CAT_FIELDS, NUMERIC_FIELDS)
flan_tm = evaluate_predictions(test_flat, flan_tp, CAT_FIELDS, NUMERIC_FIELDS)
print_metrics('Flan-T5-Large — VALID', flan_vm)
print_metrics('Flan-T5-Large — TEST', flan_tm)

## Step 8: Smart Per-Field Ensemble (TF-IDF + Flan-T5-Large)

For each of the 54 fields, pick whichever model achieves higher accuracy on the validation set. This exploits complementarity:
- **TF-IDF** excels at high-frequency categorical fields
- **Flan-T5-Large** handles complex contextual / inferential fields better

In [ ]:
def get_per_field_acc(yt_df, yp_df, fields):
    # Return a dict of {field: accuracy} on the given predictions
    accs = {}
    for col in fields:
        yt = yt_df[col].astype(str).str.strip().str.lower()
        yp = yp_df[col].astype(str).str.strip().str.lower()
        accs[col] = (yt == yp).mean()
    return accs

# Per-field accuracy for each model on validation
acc_flan  = get_per_field_acc(valid_flat, flan_vp, ALL_FIELDS)
acc_tfidf = get_per_field_acc(valid_flat, tfidf_vp, ALL_FIELDS)

# ------------------------------------------------------------------
# Build the ensemble: pick best model per field
# ------------------------------------------------------------------
smart_vp = pd.DataFrame(index=valid_flat.index)
smart_tp = pd.DataFrame(index=test_flat.index)
model_choice = {}

for col in ALL_FIELDS:
    flan_acc  = acc_flan.get(col, 0)
    tfidf_acc = acc_tfidf.get(col, 0)

    if tfidf_acc >= flan_acc:
        # TF-IDF wins (or ties — prefer simpler model on ties)
        smart_vp[col] = tfidf_vp[col]
        smart_tp[col] = tfidf_tp[col]
        model_choice[col] = f'TF-IDF ({tfidf_acc:.0%})'
    else:
        smart_vp[col] = flan_vp[col]
        smart_tp[col] = flan_tp[col]
        model_choice[col] = f'Flan-T5-Large ({flan_acc:.0%})'

# ------------------------------------------------------------------
# Summary: which model was chosen per field
# ------------------------------------------------------------------
flan_count  = sum(1 for v in model_choice.values() if 'Flan' in v)
tfidf_count = len(model_choice) - flan_count
print(f'Model selection: Flan-T5-Large chosen for {flan_count} fields, '
      f'TF-IDF chosen for {tfidf_count} fields\n')

for col in ALL_FIELDS:
    fa = acc_flan.get(col, 0)
    ta = acc_tfidf.get(col, 0)
    chosen = model_choice[col]
    print(f'  {col:60s} TF-IDF={ta:.0%}  Flan={fa:.0%}  -> {chosen}')

# ------------------------------------------------------------------
# Evaluate the ensemble
# ------------------------------------------------------------------
smart_vm = evaluate_predictions(valid_flat, smart_vp, CAT_FIELDS, NUMERIC_FIELDS)
smart_tm = evaluate_predictions(test_flat, smart_tp, CAT_FIELDS, NUMERIC_FIELDS)
print_metrics('\nSMART 2-MODEL ENSEMBLE — VALID', smart_vm)
print_metrics('SMART 2-MODEL ENSEMBLE — TEST', smart_tm)

## Step 9: Rule-Based Constraint Layer

Apply 12 hand-crafted domain rules on top of the ensemble:

1. Regex gender extraction from report text
2. Gender-gated anatomy (male fields → Not Applicable for females, vice versa)
3. Training-prior correction for rare "Not Mentioned" predictions
4. T4b consistency (non-T4b → adjacent structures default to "No")
5. is_t4a / t_stage agreement
6. MR TRG gating (only for post-treatment MRIs)
7. Post-treatment field gating (only for non-first MRIs)
8. Regex age extraction from report text
9. report_type fix (always "MRI" in this dataset)
10. MRI stage mutual exclusivity (restaging ≠ first)
11. Numeric sanity clamping (age format, extreme distances)
12. Reserved for future model-specific overrides

In [ ]:
from copy import deepcopy

# ------------------------------------------------------------------
# Snapshot BEFORE state
# ------------------------------------------------------------------
print("=" * 70)
print("RULE-BASED CONSTRAINT LAYER")
print("=" * 70)

before_vm = evaluate_predictions(valid_flat, smart_vp, CAT_FIELDS, NUMERIC_FIELDS)
before_tm = evaluate_predictions(test_flat, smart_tp, CAT_FIELDS, NUMERIC_FIELDS)
print_metrics("\n[BEFORE rules] Ensemble — VALID", before_vm)
print_metrics("[BEFORE rules] Ensemble — TEST", before_tm)

# ------------------------------------------------------------------
# Learn training-data priors for each field
# ------------------------------------------------------------------
train_distributions = {}
for col in ALL_FIELDS:
    vc = train_flat[col].astype(str).str.strip().str.lower().value_counts(normalize=True)
    train_distributions[col] = vc

# Fields where "Not Mentioned" almost never appears in training
# -> model predicting it is likely wrong; use training-mode default instead
not_mentioned_rare_fields = {}
for col in ALL_FIELDS:
    dist = train_distributions[col]
    nm_frac = dist.get('not mentioned', 0)
    if nm_frac < 0.10:
        not_mentioned_rare_fields[col] = dist.index[0]  # most common label

print(f"\nFields where 'Not Mentioned' is rare (<10% in train): "
      f"{len(not_mentioned_rare_fields)}")
for col, default in sorted(not_mentioned_rare_fields.items()):
    nm_frac = train_distributions[col].get('not mentioned', 0)
    print(f"  {col:60s} -> default='{default}' (NM={nm_frac:.0%} in train)")


# ------------------------------------------------------------------
# Regex-based extractors
# ------------------------------------------------------------------

def extract_gender_from_text(report_text):
    # Reliably extract gender using keyword patterns in Indian MRI reports
    text = str(report_text).lower()
    male_patterns = [
        r'\bmale\b', r'\b(?:mr|sri|shri)\.?\s', r'\bman\b', r'\bhis\b', r'\bhe\b',
        r'\bprostat(?:e|ic)\b', r'\bseminal\s+vesicle',
    ]
    female_patterns = [
        r'\bfemale\b', r'\b(?:mrs|ms|smt)\.?\s', r'\bwoman\b', r'\bher\b', r'\bshe\b',
        r'\buter(?:us|ine)\b', r'\bovari(?:es|an)\b', r'\bvagin(?:a|al)\b',
        r'\bcervix\b', r'\bcervical\b',
    ]
    male_score   = sum(1 for p in male_patterns   if re.search(p, text))
    female_score = sum(1 for p in female_patterns if re.search(p, text))
    if male_score > female_score:   return 'Male'
    if female_score > male_score:   return 'Female'
    return None  # uncertain

def extract_age_from_text(report_text):
    # Extract age from common patterns: '57/M', 'age: 57', '57 year old'
    text = str(report_text)
    patterns = [
        r'(\d{1,3})\s*(?:year|yr|y)[\s\-]*(?:old|age)',
        r'(?:age|aged)\s*[:\-]?\s*(\d{1,3})',
        r'(\d{1,3})\s*/\s*[MFmf]\b',
        r'\b(\d{2})\s*(?:male|female)\b',
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            age = int(m.group(1))
            if 1 <= age <= 120:
                return str(age)
    return None


# ------------------------------------------------------------------
# Main rule-application function (12 rules)
# ------------------------------------------------------------------

def apply_rules(pred_df, source_texts, train_flat_ref):
    # Apply all 12 domain-knowledge rules to fix prediction errors
    result = pred_df.copy()
    n = len(result)
    rule_counts = {}

    def count(rule_name, mask):
        rule_counts[rule_name] = rule_counts.get(rule_name, 0) + int(mask.sum())

    # RULE 1: Gender extraction from text
    for i in range(n):
        text = str(source_texts.iloc[i]) if hasattr(source_texts, 'iloc') else str(source_texts[i])
        extracted = extract_gender_from_text(text)
        if extracted is not None:
            old = result.at[result.index[i], 'bio.gender']
            if str(old).strip().lower() != extracted.lower():
                result.at[result.index[i], 'bio.gender'] = extracted

    # RULE 2: Gender-gated anatomy fields
    male_only   = ['adjacent_structures_t4b.male.prostate',
                   'adjacent_structures_t4b.male.seminal_vesicles']
    female_only = ['adjacent_structures_t4b.female.ovaries',
                   'adjacent_structures_t4b.female.uterus',
                   'adjacent_structures_t4b.female.vagina']
    gender = result['bio.gender'].astype(str).str.strip().str.lower()

    for col in female_only:
        mask = (gender == 'male') & (result[col].astype(str).str.strip().str.lower() != 'not applicable')
        result.loc[mask, col] = 'Not Applicable'
        count('gender_gate_female_fields', mask)
    for col in male_only:
        mask = (gender == 'female') & (result[col].astype(str).str.strip().str.lower() != 'not applicable')
        result.loc[mask, col] = 'Not Applicable'
        count('gender_gate_male_fields', mask)

    # RULE 3: Fix "Not Mentioned" for fields where it's rare in training
    for col, default_val in not_mentioned_rare_fields.items():
        proper_case = train_flat_ref[col].astype(str).value_counts().index[0]
        pred_lower  = result[col].astype(str).str.strip().str.lower()
        mask = (pred_lower == 'not mentioned')
        result.loc[mask, col] = proper_case
        count(f'nm_replace_{col}', mask)

    # RULE 4: T4b adjacent-structure gating
    t4b_adj_fields = [
        'adjacent_structures_t4b.male_and_female.puborectalis',
        'adjacent_structures_t4b.male_and_female.levator_ani',
        'adjacent_structures_t4b.male_and_female.obturator_internus',
        'adjacent_structures_t4b.male_and_female.obturator_externus',
        'adjacent_structures_t4b.male_and_female.piriformis',
    ] + male_only + female_only

    t_stage_lower = result['t_stage'].astype(str).str.strip().str.lower()
    not_t4b = (t_stage_lower != 't4b')

    for col in t4b_adj_fields:
        pred_lower = result[col].astype(str).str.strip().str.lower()
        mask = not_t4b & (pred_lower == 'yes')
        train_yes_rate = train_distributions.get(col, pd.Series()).get('yes', 0)
        if train_yes_rate < 0.15:
            result.loc[mask, col] = 'No'
            count('t4b_gate_adj', mask)

    # RULE 5: is_t4a <-> t_stage consistency
    is_t4a     = t_stage_lower == 't4a'
    is_not_t4a = ~t_stage_lower.isin(['t4a', 'not mentioned'])

    mask_yes = is_t4a & (result['is_t4a'].astype(str).str.strip().str.lower() != 'yes')
    result.loc[mask_yes, 'is_t4a'] = 'Yes'
    count('t4a_consistency_yes', mask_yes)

    mask_no = is_not_t4a & (result['is_t4a'].astype(str).str.strip().str.lower() == 'yes')
    result.loc[mask_no, 'is_t4a'] = 'No'
    count('t4a_consistency_no', mask_no)

    # RULE 6: MR TRG only for post-treatment / restaging MRIs
    is_post_tx   = result['mri_stage.is_post_treatment_mri_report'].astype(str).str.strip().str.lower()
    is_restaging = result['mri_stage.is_restaging_mri_report'].astype(str).str.strip().str.lower()
    not_post     = (is_post_tx != 'yes') & (is_restaging != 'yes')
    mr_trg_lower = result['mr_trg'].astype(str).str.strip().str.lower()
    mask = not_post & (~mr_trg_lower.isin(['not mentioned', 'not applicable']))
    result.loc[mask, 'mr_trg'] = 'Not Mentioned'
    count('mr_trg_gate', mask)

    # RULE 7: Post-treatment change fields only for post-tx MRIs
    post_tx_fields = [
        'post_treatment_change.is_thick_t2_hypointense_band',
        'post_treatment_change.is_thin_t2_hypointense_band',
        'post_treatment_change.is_residual_tumor_as_first_mri',
        'post_treatment_change.is_mucin_reaction_t2_hyper_hypo_post_treatment',
    ]
    is_first      = result['mri_stage.is_first_mri_report'].astype(str).str.strip().str.lower()
    clearly_first = (is_first == 'yes') & (is_post_tx != 'yes')
    for col in post_tx_fields:
        pred_lower = result[col].astype(str).str.strip().str.lower()
        mask = clearly_first & (~pred_lower.isin(['not mentioned', 'not applicable']))
        result.loc[mask, col] = 'Not Mentioned'
        count('post_tx_gate', mask)

    # RULE 8: Age extraction from text
    for i in range(n):
        text = str(source_texts.iloc[i]) if hasattr(source_texts, 'iloc') else str(source_texts[i])
        extracted_age = extract_age_from_text(text)
        if extracted_age is not None:
            current   = str(result.at[result.index[i], 'bio.age']).strip()
            curr_nums = re.findall(r'\d+', current)
            if not curr_nums or abs(int(curr_nums[0]) - int(extracted_age)) > 5:
                result.at[result.index[i], 'bio.age'] = extracted_age

    # RULE 9: report_type is always "MRI" for this dataset
    mask = result['report_type'].astype(str).str.strip().str.lower() != 'mri'
    result.loc[mask, 'report_type'] = 'MRI'
    count('report_type_fix', mask)

    # RULE 10: MRI stage mutual exclusivity
    restaging_yes = result['mri_stage.is_restaging_mri_report'].astype(str).str.strip().str.lower() == 'yes'
    first_not_no  = result['mri_stage.is_first_mri_report'].astype(str).str.strip().str.lower() != 'no'
    mask = restaging_yes & first_not_no
    result.loc[mask, 'mri_stage.is_first_mri_report'] = 'No'
    count('restaging_implies_not_first', mask)

    # RULE 11: Numeric sanity clamping
    # bio.age: strip erroneous "mm" suffix
    for i in range(n):
        age_val = str(result.at[result.index[i], 'bio.age']).strip()
        if 'mm' in age_val.lower():
            nums = re.findall(r'\d+', age_val)
            if nums:
                result.at[result.index[i], 'bio.age'] = nums[0]

    # Distance fields: clamp values >200mm as suspicious
    distance_fields = [
        'mri_numeric.distance_from_anal_verge',
        'mri_numeric.distance_from_anorectal_junction',
        'mri_numeric.extramural_spread_size',
        'mri_numeric.mr_crm_distance',
    ]
    for col in distance_fields:
        for i in range(n):
            val  = str(result.at[result.index[i], col]).strip()
            nums = re.findall(r'\d+(?:\.\d+)?', val)
            if nums:
                v = float(nums[0])
                if 'cm' in val.lower():
                    v *= 10
                if v > 200:
                    result.at[result.index[i], col] = 'Not Mentioned'

    # --- Summary ---
    print("\n--- Rule application summary ---")
    total_fixes = 0
    for rule, cnt in sorted(rule_counts.items(), key=lambda x: -x[1]):
        if cnt > 0:
            print(f"  {rule}: {cnt} corrections")
            total_fixes += cnt
    print(f"  TOTAL corrections: {total_fixes}")
    return result


# ------------------------------------------------------------------
# Apply rules to both splits
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("Applying rules to VALIDATION set...")
print("=" * 70)
ruled_vp = apply_rules(smart_vp, valid_flat['text'], train_flat)

print("\n" + "=" * 70)
print("Applying rules to TEST set...")
print("=" * 70)
ruled_tp = apply_rules(smart_tp, test_flat['text'], train_flat)

# ------------------------------------------------------------------
# Evaluate after rules
# ------------------------------------------------------------------
ruled_vm = evaluate_predictions(valid_flat, ruled_vp, CAT_FIELDS, NUMERIC_FIELDS)
ruled_tm = evaluate_predictions(test_flat, ruled_tp, CAT_FIELDS, NUMERIC_FIELDS)

print("\n" + "=" * 70)
print("RESULTS AFTER RULE-BASED CORRECTIONS")
print("=" * 70)
print_metrics("\n[AFTER rules] Ensemble + Rules — VALID", ruled_vm)
print_metrics("[AFTER rules] Ensemble + Rules — TEST", ruled_tm)

# Improvement delta
print("\n--- Improvement ---")
for metric in ['leaf_avg_cat_accuracy', 'micro_cat_accuracy', 'leaf_avg_macro_f1']:
    bv = before_vm[metric]; av = ruled_vm[metric]
    bt = before_tm[metric]; at_ = ruled_tm[metric]
    print(f"  VALID {metric}: {bv:.4f} -> {av:.4f} ({av - bv:+.4f})")
    print(f"  TEST  {metric}: {bt:.4f} -> {at_:.4f} ({at_ - bt:+.4f})")

## Step 10: Per-Field Error Analysis & Remaining Errors

In [ ]:
# ------------------------------------------------------------------
# Per-field accuracy comparison: before vs after rules
# ------------------------------------------------------------------
err_before = per_field_accuracy(valid_flat, smart_vp, ALL_FIELDS)
err_after  = per_field_accuracy(valid_flat, ruled_vp, ALL_FIELDS)

comparison = err_before.merge(err_after, on='field', suffixes=('_before', '_after'))
comparison['delta'] = comparison['accuracy_after'] - comparison['accuracy_before']
comparison = comparison.sort_values('delta', ascending=False)

print("=" * 70)
print("PER-FIELD ACCURACY CHANGES (improved fields)")
print("=" * 70)
improved = comparison[comparison['delta'] > 0]
for _, row in improved.iterrows():
    print(f"  {row['field']:60s} {row['accuracy_before']:.0%} -> {row['accuracy_after']:.0%} "
          f"({row['delta']:+.0%}, {row['n_errors_before'] - row['n_errors_after']:+d} errors fixed)")

worsened = comparison[comparison['delta'] < 0]
if len(worsened) > 0:
    print(f"\nWORSENED fields ({len(worsened)}):")
    for _, row in worsened.iterrows():
        print(f"  {row['field']:60s} {row['accuracy_before']:.0%} -> {row['accuracy_after']:.0%} "
              f"({row['delta']:+.0%})")

# ------------------------------------------------------------------
# Remaining worst fields
# ------------------------------------------------------------------
err_final = per_field_accuracy(valid_flat, ruled_vp, ALL_FIELDS).sort_values('accuracy')

print("\n" + "=" * 70)
print("REMAINING WORST FIELDS (after rules)")
print("=" * 70)
print(err_final.head(15).to_string(index=False))

n100 = (err_final['accuracy'] == 1.0).sum()
n90  = (err_final['accuracy'] >= 0.9).sum()
n80  = (err_final['accuracy'] >= 0.8).sum()
print(f"\nFields at 100%: {n100}/{len(ALL_FIELDS)}")
print(f"Fields at >=90%: {n90}/{len(ALL_FIELDS)}")
print(f"Fields at >=80%: {n80}/{len(ALL_FIELDS)}")
print(f"Average field accuracy: {err_final['accuracy'].mean():.3f}")

# ------------------------------------------------------------------
# Show specific remaining errors for debugging
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("SPECIFIC REMAINING ERRORS (worst 10 fields)")
print("=" * 70)
for field in err_final.head(10)['field'].tolist():
    yt = valid_flat[field].astype(str).tolist()
    yp = ruled_vp[field].astype(str).tolist()
    errors = [(t, p) for t, p in zip(yt, yp)
              if t.strip().lower() != p.strip().lower()]
    if errors:
        print(f"\n{field} ({len(errors)} errors):")
        for t, p in errors[:5]:
            print(f"  TRUE: {t!r:40s} PRED: {p!r}")

## Step 11: Final Comparison & Save Results

In [ ]:
# ------------------------------------------------------------------
# Final comparison table
# ------------------------------------------------------------------
print("=" * 70)
print("FINAL COMPARISON TABLE")
print("=" * 70)

final_results = pd.DataFrame([
    fmt('TF-IDF',                'VALID', tfidf_vm),
    fmt('TF-IDF',                'TEST',  tfidf_tm),
    fmt('Flan-T5-Large',         'VALID', flan_vm),
    fmt('Flan-T5-Large',         'TEST',  flan_tm),
    fmt('2-Model Ensemble',      'VALID', smart_vm),
    fmt('2-Model Ensemble',      'TEST',  smart_tm),
    fmt('Ensemble + Rules',      'VALID', ruled_vm),
    fmt('Ensemble + Rules',      'TEST',  ruled_tm),
])
display(final_results)

# ------------------------------------------------------------------
# Save all outputs
# ------------------------------------------------------------------
OUT = '/content/results_2model_ensemble'
os.makedirs(OUT, exist_ok=True)

# Predictions
smart_vp.to_csv(f'{OUT}/ensemble_valid_pred.csv', index=False)
smart_tp.to_csv(f'{OUT}/ensemble_test_pred.csv', index=False)
ruled_vp.to_csv(f'{OUT}/ruled_ensemble_valid_pred.csv', index=False)
ruled_tp.to_csv(f'{OUT}/ruled_ensemble_test_pred.csv', index=False)
flan_vp.to_csv(f'{OUT}/flan_large_valid_pred.csv', index=False)
flan_tp.to_csv(f'{OUT}/flan_large_test_pred.csv', index=False)

# Analysis
err_final.to_csv(f'{OUT}/per_field_accuracy.csv', index=False)
comparison.to_csv(f'{OUT}/per_field_before_after_rules.csv', index=False)
final_results.to_csv(f'{OUT}/final_comparison.csv', index=False)

# Model choice log
pd.DataFrame([{'field': k, 'chosen_model': v} for k, v in model_choice.items()]
            ).to_csv(f'{OUT}/model_choice_per_field.csv', index=False)

print(f'\nAll results saved to {OUT}')

# Download on Colab
try:
    from google.colab import files
    for f in os.listdir(OUT):
        files.download(f'{OUT}/{f}')
except:
    pass